## 6 — MRdeeP (state-level estimates, CES sample3 / 5000)
Multivariate Multilevel Regression with Deep Generative Post-Stratification.
Outcomes extracted: `climate_problem`, `renewable_fuel`.

Pipeline:
1. `insert_data` — encodes CES survey + county-level benchmark
2. `fit` — trains an ensemble of CGANs (Wasserstein loss + gradient penalty)
3. `post_stratify('state_fips')` — generates synthetic micro-data per demographic
   cell, groups by state → extracts `climate_problem` and `renewable_fuel` estimates

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from pathlib import Path

os.environ['DEEPVERSE_BACKEND'] = 'pytorch'
sys.path.insert(0, '/Users/carmenk/Documents/CSS/Capstone/mrdeep/python')
from mrdeep import MRdeeP

sys.path.insert(0, str(Path('.').resolve()))
from utils import OUTPUT_DIR, STATE_FIPS_TO_NAME, SURVEY_PATH, save_estimates

DATA_DIR   = Path("../../")
OUTCOME    = ['climate_problem', 'renewable_fuel']
MODEL_NAME = 'mrdeep'

### 1. Load and prepare data

In [2]:
OUTCOME_COLS = ['climate_problem','regulate_carbon','renewable_fuel',
                'clean_air_water','fuel_efficiency','fossil_fuel','paris_agreement']
DEMOG_VARS   = ['gender', 'race4', 'educ_category', 'county_fips', 'state_fips']

raw       = pd.read_csv(SURVEY_PATH, dtype={'state_fips': str, 'county_fips': str})
ps_county = pd.read_csv(DATA_DIR / 'post_stratification_frame' / 'poststrat_county.csv',
                        dtype={'state_fips': str, 'county_fips': str})

raw['county_fips'] = raw['county_fips'].astype(str).str.zfill(5)
OUTCOME_COLS = [c for c in OUTCOME_COLS if c in raw.columns]

survey = raw[DEMOG_VARS + OUTCOME_COLS].dropna().copy()
survey['educ_category'] = survey['educ_category'].astype(str)

benchmark = ps_county[DEMOG_VARS + ['N_rounded']].copy()
benchmark['educ_category'] = benchmark['educ_category'].astype(str)
target_rows = len(benchmark)
benchmark['count'] = np.maximum(
    1,
    (benchmark['N_rounded'] / benchmark['N_rounded'].sum() * target_rows).round(),
).astype(int)
benchmark = benchmark.drop(columns=['N_rounded'])

print(f'Survey (complete cases): {len(survey):,}')
for oc in OUTCOME_COLS:
    print(f'  {oc}: {survey[oc].mean()*100:.1f}% support')
print(f'Benchmark strata: {len(benchmark):,}  augmented rows: {benchmark["count"].sum():,}')

Survey (complete cases): 4,962
  climate_problem: 64.4% support
  regulate_carbon: 65.9% support
  renewable_fuel: 61.0% support
  clean_air_water: 57.6% support
  fuel_efficiency: 66.8% support
  fossil_fuel: 63.0% support
  paris_agreement: 60.5% support
Benchmark strata: 99,940  augmented rows: 170,690


### 2. Insert data into MRdeeP

In [3]:
mod = MRdeeP(ensembles=3, random_state=42)

mod.insert_data(
    survey     = survey,
    benchmark  = benchmark,
    demog_vars = DEMOG_VARS,
    count_col  = 'count',
    oversample = 1,
)
print(mod)

MRdeeP (backend=pytorch, ensembles=3)
  Data inserted: True
  Survey: 4962 obs, 7 substantive vars, 5 demographic vars
  Augmented benchmark: 170690 rows
  Fitted: False


### 3. Train CGAN ensemble
Default architecture: 4 × 256-neuron hidden layers, Wasserstein loss + gradient penalty.

In [4]:
mod.fit(
    epochs        = 500,
    patience      = 50,
    batch_size    = 256,
    k             = 32,
    print_runtime = True,
)
print(mod)

Ensemble 1/3


Ensemble 2/3


Ensemble 3/3


Total fit time: 475.3 seconds.
MRdeeP (backend=pytorch, ensembles=3)
  Data inserted: True
  Survey: 4962 obs, 7 substantive vars, 5 demographic vars
  Augmented benchmark: 170690 rows
  Fitted: True
    Noise dim (k): 32
    Ensemble members: 3
    Generated survey: 170690 rows
    Total fit time: 475.3s


### 4. Post-stratify → state-level estimates for all outcomes

In [5]:
estimates = mod.post_stratify(levels='state_fips')
print(f'Estimates shape: {estimates.shape}  ({estimates["state_fips"].nunique()} states)')
estimates.head()

Estimates shape: (51, 8)  (51 states)


,state_fips,clean_air_water,climate_problem,fossil_fuel,fuel_efficiency,paris_agreement,regulate_carbon,renewable_fuel
0,01,0.613182,0.612046,0.587091,0.734248,0.553761,0.637524,0.690710
1,02,0.609184,0.599957,0.597240,0.733751,0.549566,0.634882,0.689475
2,04,0.587969,0.580066,0.615282,0.715835,0.527427,0.608530,0.666020
3,05,0.601522,0.600547,0.593818,0.725997,0.538732,0.626582,0.680315
4,06,0.631832,0.622687,0.585977,0.747338,0.565594,0.655050,0.702845


### 5. Extract target outcomes and save

In [6]:
for OUTCOME_VAR in OUTCOME:
    result = estimates[['state_fips', OUTCOME_VAR]].rename(
        columns={OUTCOME_VAR: 'estimate'}
    ).copy()
    result['state_name'] = result['state_fips'].map(STATE_FIPS_TO_NAME)

    save_estimates(result, MODEL_NAME, OUTCOME_VAR)

    print(f'\n--- {OUTCOME_VAR} ---')
    print(f'National mean: {result["estimate"].mean():.3f}')
    print(result.sort_values("estimate", ascending=False).head(5)[["state_name","estimate"]].to_string(index=False))


  mrdeep (climate_problem) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.6075
  Median estimate:       0.6096
  Min estimate:          0.5801
  Max estimate:          0.6227

  Saved → /Users/kaeleyoshea/Capstone/A.MRdeeP-Deep-Learning--MRP/model_run_ces/sample3_state/outputs/estimates/climate_problem_state_estimates.csv


--- climate_problem ---
National mean: 0.608
   state_name  estimate
   California  0.622687
 Pennsylvania  0.619549
     Illinois  0.618896
     Delaware  0.617650
New Hampshire  0.616076

  mrdeep (renewable_fuel) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.6902
  Median estimate:       0.6909
  Min estimate:          0.6660
  Max estimate:          0.7031

  Saved → /Users/kaeleyoshea/Capstone/A.MRdeeP-Deep-Learning--MRP/model_run_ces/sample3_state/outputs/estimates/renewable_fuel_state_estimates.csv


--- renewable_fuel ---
National mean: 0.6